In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

test_data  = load_jsonl('datasets/QQP/test_full.jsonl')
val_data   = load_jsonl('datasets/QQP/valid.jsonl')
train_data = load_jsonl('datasets/QQP/train.jsonl')

test_df  = pd.DataFrame(test_data)
val_df   = pd.DataFrame(val_data)
train_df = pd.DataFrame(train_data)


(144715, 2)


,src,trg
0,Academic and Educational Advice: What can I do...,What should I do after bcom?
1,What are some good songs to make a texting lyr...,What is a good prank lyric text to text to a f...
2,How do I kill rats?,How can I kill a pack rat?
3,A ball is thrown vertically upward direction w...,A ball is thrown in a vertically upward direct...
4,How did Donald trump win the elections?,How did Donald Trump win the 2016 Presidential...


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=True))

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df['src_tokens'] = df['src'].apply(count_tokens)
    df['trg_tokens'] = df['trg'].apply(count_tokens)

# filter: keep rows where BOTH src and trg are at most 62 tokens
train_filtered = train_df[(train_df['src_tokens'] <= 62) & (train_df['trg_tokens'] <= 62)]
val_filtered   = val_df[(val_df['src_tokens'] <= 62) & (val_df['trg_tokens'] <= 62)]
test_filtered  = test_df[(test_df['src_tokens'] <= 62) & (test_df['trg_tokens'] <= 62)]

print(f"Length of train before: {len(train_df)}, now: {len(train_filtered)}")
print(f"Length of validation before: {len(val_df)}, now: {len(val_filtered)}")
print(f"Length of test before: {len(test_df)}, now: {len(test_filtered)}")

/lustre/uc3m/gts_c3_cluster_1-12/zualikha/envs/diffuseq/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Length of train before: 144715, now: 144697
Length of validation before: 2048, now: 2048
Length of test before: 2500, now: 2500


In [7]:
import os

def save_jsonl(df, path, cols=['src', 'trg']):
    with open(path, 'w') as f:
        for row in df[cols].to_dict(orient='records'):
            f.write(json.dumps(row) + '\n')

base = 'datasets/QQP'

os.rename(f'{base}/train.jsonl',    f'{base}/train_full.jsonl')
os.rename(f'{base}/valid.jsonl',    f'{base}/valid_full.jsonl')
os.rename(f'{base}/test_full.jsonl',f'{base}/test_full.jsonl')  # already named correctly

save_jsonl(train_filtered, f'{base}/train.jsonl')
save_jsonl(val_filtered,   f'{base}/valid.jsonl')
save_jsonl(test_filtered,  f'{base}/test.jsonl')

print("Done!")

Done!
